In [2]:
from pathlib import Path
import json
import csv
import matplotlib.pyplot as plt
import numpy as np

# Individua la cartella del progetto anche se il kernel parte da un'altra directory.
project_dir = Path.cwd()
if not (project_dir / "results" / "stress_from_channel").exists():
    project_dir = Path(r"C:\Users\RHEODLS\Documents\Luca\script_rheoflu")

shape_csv = project_dir / "results" / "stress_from_channel" / "stress_profile_space.csv"
piv_csv = project_dir / "results" / "piv" / "extensional_stress_profile.csv"
shape_json = project_dir / "results" / "stress_from_channel" / "extensional_stress_from_channel_shape.json"
piv_json = project_dir / "results" / "piv" / "extensional_stress_profile.json"

def load_plot_component(csv_path, array_key):
    with csv_path.open(encoding="utf-8-sig", newline="") as data_file:
        rows = [row for row in csv.DictReader(data_file) if row["array_key"] == array_key]
    rows.sort(key=lambda row: int(row["index_0"]))
    return np.asarray([float(row["value"]) for row in rows])

x_shape_um = load_plot_component(shape_csv, "axes_1_line_0_x")
stress_shape_pa = load_plot_component(shape_csv, "axes_1_line_0_y")
x_piv_um = load_plot_component(piv_csv, "axes_1_line_0_x")
stress_piv_pa = load_plot_component(piv_csv, "axes_1_line_0_y")
with shape_json.open(encoding="utf-8") as file:
    shape_meta = json.load(file)
with piv_json.open(encoding="utf-8") as file:
    piv_meta = json.load(file)

if len(x_shape_um) != len(stress_shape_pa) or len(x_piv_um) != len(stress_piv_pa):
    raise ValueError("Ogni profilo deve avere coordinate x e stress della stessa lunghezza.")
if shape_meta["x_units"] != piv_meta["x_units"] or shape_meta["stress_units"] != piv_meta["stress_units"]:
    raise ValueError("Le unità dei due profili non coincidono.")

shape_source = shape_meta["analysis_parameters"]["channel_image"]
piv_source = piv_meta["analysis_parameters"]["input_dir"]
same_folder = Path(shape_source).parent == Path(piv_source)

output_dir = project_dir / "results" / "technique_comparison"
output_dir.mkdir(parents=True, exist_ok=True)
plot_path = output_dir / "stress_methods_comparison.png"
data_path = plot_path.with_suffix(".csv")
metadata_path = plot_path.with_suffix(".json")

fig, ax = plt.subplots(figsize=(13, 5.5))
ax.plot(x_shape_um, stress_shape_pa, color="crimson", linewidth=2.2, label="From channel shape")
ax.plot(x_piv_um, 0.00007*stress_piv_pa, color="darkblue", marker="o", markersize=3, linewidth=1.8, label="PIV")
ax.axhline(0, color="gray", linestyle=":", linewidth=1.1)
ax.set_xlabel(f"x [{shape_meta['x_units']}]")
ax.set_ylabel(f"Extensional stress [{shape_meta['stress_units']}]")
ax.set_title("Extensional stress: PIV vs From channel shape")
ax.set_yscale("symlog", linthresh=0.01)
ax.grid(True, which="both", alpha=0.25)
ax.legend()
ax.text(0.01, 0.02, source_note, transform=ax.transAxes, fontsize=9, color="dimgray")
fig.tight_layout()
fig.savefig(plot_path, dpi=300, bbox_inches="tight")

with data_path.open("w", encoding="utf-8-sig", newline="") as data_file:
    writer = csv.DictWriter(data_file, fieldnames=["method", "x_um", "stress_pa"])
    writer.writeheader()
    for x_value, stress_value in zip(x_shape_um, stress_shape_pa):
        writer.writerow({"method": "From channel shape", "x_um": x_value, "stress_pa": stress_value})
    for x_value, stress_value in zip(x_piv_um, stress_piv_pa):
        writer.writerow({"method": "PIV", "x_um": x_value, "stress_pa": stress_value})
comparison_metadata = {
    "plot_type": "stress_method_comparison",
    "title": "Extensional stress: PIV vs From channel shape",
    "x_units": shape_meta["x_units"],
    "stress_units": shape_meta["stress_units"],
    "y_scale": "symlog",
    "symlog_linthresh_pa": 0.01,
    "same_dataset_folder": same_folder,
    "source_data": {
        "from_channel_shape_csv": str(shape_csv),
        "from_channel_shape_metadata": str(shape_json),
        "piv_csv": str(piv_csv),
        "piv_metadata": str(piv_json),
    },
    "numeric_data_file": data_path.name,
    "series": {
        "from_channel_shape": {"x_key": "from_channel_shape_x_um", "y_key": "from_channel_shape_stress_pa", "n_points": int(len(x_shape_um))},
        "piv": {"x_key": "piv_x_um", "y_key": "piv_stress_pa", "n_points": int(len(x_piv_um))},
    },
}
with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(comparison_metadata, file, indent=2, ensure_ascii=False)

print(f"Grafico salvato: {plot_path.resolve()}")
print(f"Dati numerici salvati: {data_path.resolve()}")
print(f"Metadati salvati: {metadata_path.resolve()}")
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/lucaasinari/Desktop/Tesi ESPCI/script_rheoflu/results/piv/extensional_stress_profile.csv'